# 02.5 Binding

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Model attributes, component arrays, a simple function, curve fitting and plotting.

**Introduces:** `Linear` and `Langmuir` binding, equilibrium, batch uptake data, isotherm fitting, kinetics, competition and capacity.

1. Linear binding
2. Batch uptake data, isotherm fit and simulation
3. Binding in a column
4. Competitive Langmuir binding

Binding retains components on the stationary phase. We first isolate it in a closed batch tank, then add it to the tracer column from First Model.

In [ ]:
%matplotlib inline

An adsorption isotherm describes the equilibrium relation between the two phases:

```{note}
**Isotherm:**
The equilibrium relation between liquid concentration $c$ and bound concentration $q$ at a fixed temperature.
```

```{figure} ./resources/isotherm.png
:width: 50%
:align: center
```

CADET-Process provides binding models with different levels of detail. Here, Linear introduces binding strength and kinetics; Langmuir adds finite capacity and competition.

See the [CADET-Process binding model guide](https://cadet-process.readthedocs.io/en/latest/user_guide/process_model/binding_model.html) for how binding models are set up, and the [CADET-Core binding model overview](https://cadet.github.io/master/modelling/binding/index.html) for the equations of all available models.

```{figure} ./resources/isotherm_models.png
:width: 100%
:align: center
```

## Example 1: Linear model

The simplest model for adsorption is the linear model ([equations in CADET-Core](https://cadet.github.io/master/modelling/binding/linear.html), [`Linear` in CADET-Process](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Linear.html)).

The liquid concentration $c$ and bound concentration $q$ are two **model states**. Linear binding transfers material between them:

$$
\frac{dq}{dt} = k_a c - k_d q,
\qquad q_{\mathrm{eq}} = \frac{k_a}{k_d}c.
$$

`adsorption_rate` sets $k_a$, `desorption_rate` sets $k_d$. With `is_kinetic=True`, we follow the transfer over time; `False` assumes binding is already at equilibrium. Binding models are kinetic by default; [99.6 Common Problems](../../99%20Misc/99.6%20Common%20Problems/) shows when this matters.

A closed `Cstr` represents a batch uptake experiment with **1 L total volume** and equal liquid and solid volumes (**porosity 0.5**). The liquid starts at **1 mol/m³**; the resin is initially unloaded.

We create one component and then a `Linear` binding model for the tank.

In [ ]:
from CADETProcess.processModel import ComponentSystem

component_system = ComponentSystem(1)

The binding model is created with `Linear(component_system, name="linear")`; printing its `parameters` shows the settings and defaults. We set:

| Attribute | Value |
|---|---|
| `is_kinetic` | `True` |
| `adsorption_rate` | `[2]` ($k_a$) |
| `desorption_rate` | `[1]` s⁻¹ ($k_d$) |

In [ ]:
# Linear binding model


`init_liquid_volume` and `const_solid_volume` set the two phase volumes.

In [ ]:
import numpy as np
from CADETProcess.processModel import  Cstr

reactor = Cstr(component_system, name='reactor')
reactor.binding_model = binding_model  # Attach the binding model configured above.

reactor.init_liquid_volume = 0.5*1e-3
reactor.const_solid_volume = 0.5*1e-3

`c` sets the initial liquid concentration; `q` sets the initial bound concentration.

In [ ]:
reactor.c = [1]

Recording both states shows the transfer: `write_solution_bulk` saves liquid concentration $c$ as `sim_results.solution.reactor.bulk`; `write_solution_solid` saves bound concentration $q$ as `sim_results.solution.reactor.solid`. The plots show where the component is, while the total amount stays constant in this closed tank.

In [ ]:
reactor.solution_recorder.write_solution_bulk = True
reactor.solution_recorder.write_solution_solid = True

The tank goes into a `FlowSheet`, and a `Process` for the uptake duration is simulated with `Cadet`.

In [ ]:
from CADETProcess.processModel import FlowSheet

flow_sheet = FlowSheet(component_system)

flow_sheet.add_unit(reactor)

In [ ]:
from CADETProcess.processModel import Process

process = Process(flow_sheet, 'process')
process.cycle_time = 10

In [ ]:
from CADETProcess.simulator import Cadet
simulator = Cadet()
sim_results = simulator.simulate(process)

_ = sim_results.solution.reactor.bulk.plot()
_ = sim_results.solution.reactor.solid.plot()

### Resolve the uptake over time

Binding is fast compared with the default **1 s** output spacing. A smaller `simulator.time_resolution` records the transient more clearly.

In [ ]:
simulator.time_resolution = 0.1

The finer output spacing reveals the approach to equilibrium.

In [ ]:
sim_results = simulator.simulate(process)

_ = sim_results.solution.reactor.bulk.plot()
_ = sim_results.solution.reactor.solid.plot()

## Example 2: from batch uptake data to a binding model

In a **batch uptake experiment**, mix a known liquid volume with a known amount of resin, wait for equilibrium and measure the remaining liquid concentration $c_{eq}$. The amount missing from the liquid is bound to the resin.

With initially unloaded resin, $q_{eq}=(c_0-c_{eq})V_{liquid}/V_{solid}$. Here the two volumes are equal, as in our batch tank, so $q_{eq}=c_0-c_{eq}$.

```{note}
These are synthetic equilibrium data for demonstrating the workflow. Liquid concentrations are per liquid volume; bound concentrations are per solid volume. Both use mol/m³.
```

In [ ]:
# Initial and measured equilibrium liquid concentrations from five batch experiments.
import matplotlib.pyplot as plt

c_initial = np.array([0.302, 0.748, 1.510, 2.970, 4.530])
c_equilibrium = np.array([0.10, 0.25, 0.50, 1.00, 1.50])
q_equilibrium = c_initial - c_equilibrium

### Fit the isotherm with `curve_fit()`

The Linear isotherm is $q_{eq}=Kc_{eq}$. We write it as a function `linear_isotherm(c, K)` and fit its slope $K$ to the batch data with [`curve_fit(linear_isotherm, c_equilibrium, q_equilibrium)`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html). It returns the fitted parameters first; we store the slope as `K`, then overlay the fitted line and the data.

In [ ]:
# Fit the linear isotherm


In [ ]:
# Compare the batch data with the fitted equilibrium isotherm.
c_plot = np.linspace(0, c_equilibrium.max(), 100)
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(c_equilibrium, q_equilibrium, label="Batch uptake data")
ax.plot(c_plot, linear_isotherm(c_plot, K), label=f"Linear fit, K = {K:.2f}")
ax.set_xlabel("Equilibrium liquid concentration / mol/m³")
ax.set_ylabel("Bound concentration / mol/m³")
ax.legend()
fig.tight_layout()
plt.show()

### Use the fit in CADET-Process

For `Linear`, $K=k_a/k_d$. We reuse the batch tank from Example 1 and change its `reactor.binding_model`, then reset the initial conditions for a fresh experiment:

| Attribute | Value |
|---|---|
| `reactor.binding_model.adsorption_rate` | `[K]` (fitted) |
| `reactor.binding_model.desorption_rate` | `[1]` s⁻¹ |
| `reactor.c` | `[1]` mol/m³ |
| `reactor.q` | `[0]` mol/m³ |

```{note}
Equilibrium data determine the ratio of the rates, not how quickly binding occurs. The desorption rate of 1 s⁻¹ is an illustrative choice; uptake measurements over time would be needed to fit the kinetics.
```

In [ ]:
# Apply the fitted equilibrium factor and start a fresh batch uptake experiment.
reactor.binding_model.adsorption_rate = [K]
reactor.binding_model.desorption_rate = [1]
reactor.c = [1]
reactor.q = [0]

In [ ]:
# Simulate uptake with the fitted binding model and show both concentration states.
batch_results = simulator.simulate(process)
_ = batch_results.solution.reactor.bulk.plot()
_ = batch_results.solution.reactor.solid.plot()
plt.show()

The time plots show uptake approaching equilibrium; the isotherm plot shows the equilibrium relation used to parameterize the model.

## Example 3: binding in a column

The same `Linear` model can be attached to a column. We reuse the tracer column from First Model (**10 × 2 cm**, porosity **0.4**, dispersion **1e-7 m²/s**, **6 mL/min**) and feed a nonbinding **Tracer** together with a binding **Protein**, both at **1 mol/m³**.

In the column, the bound phase stores part of each binding component, so its front arrives later. With equilibrium binding, $K=k_a/k_d$, the retention time becomes

$$
t_R = \frac{[\varepsilon_t + (1-\varepsilon_t)K]\,V}{Q}.
$$

For the tracer, $K=0$ gives the First Model result $t_R=\varepsilon_t V/Q$. $(1-\varepsilon_t)$ is the solid fraction of the column.

In [ ]:
# The familiar tracer column, now with two components and without binding yet.
from CADETProcess.processModel import Inlet, LumpedRateModelWithoutPores, Outlet

column_component_system = ComponentSystem(["Tracer", "Protein"])
column_inlet = Inlet(column_component_system, "inlet")
column_inlet.c = [1, 1]
column_inlet.flow_rate = 1e-7
column = LumpedRateModelWithoutPores(column_component_system, "column")
column.length = 0.1
column.diameter = 0.02
column.total_porosity = 0.4
column.axial_dispersion = 1e-7
column_outlet = Outlet(column_component_system, "outlet")

column_sheet = FlowSheet(column_component_system, "flow_sheet")
column_sheet.add_unit(column_inlet)
column_sheet.add_unit(column)
column_sheet.add_unit(column_outlet)
column_sheet.add_connection(column_inlet, column)
column_sheet.add_connection(column, column_outlet)
column_process = Process(column_sheet, "column_binding")
column_process.cycle_time = 1200

We create `column_binding = Linear(column_component_system, name="column_linear")` with one entry per component (Tracer, Protein): the tracer does not bind, the protein has $K = 1.5$. As for the tank, `column.binding_model = column_binding` attaches it.

| Attribute | Value |
|---|---|
| `is_kinetic` | `False` |
| `adsorption_rate` | `[0, 1.5]` |
| `desorption_rate` | `[1, 1]` |

In [ ]:
# Linear binding for the column
column_binding = Linear(column_component_system, name="column_linear")
column_binding.is_kinetic = False
column_binding.adsorption_rate = [0, 1.5]  # Tracer, Protein
column_binding.desorption_rate = [1, 1]
column.binding_model = column_binding

The dashed lines mark the predicted retention times. Python plots show time in minutes.

In [ ]:
# Predict both retention times, then simulate and compare the outlet curves.
column_volume = np.pi * 0.01**2 * 0.1
t_tracer = 0.4 * column_volume / 1e-7
t_protein = (0.4 + 0.6 * 1.5) * column_volume / 1e-7
print(f"Predicted t_R: Tracer {t_tracer:.0f} s, Protein {t_protein:.0f} s")

column_results = Cadet().simulate(column_process)
fig, ax = column_results.solution.outlet.outlet.plot()
ax.axvline(t_tracer / 60, color="C0", linestyle="--")
ax.axvline(t_protein / 60, color="C1", linestyle="--")
plt.show()

Both components enter together; binding delays the protein front from about **126 s** to about **408 s**, while the tracer is unchanged. The exercises use this setting to compare binding strengths and capacity.

## Example 4: multi-component Langmuir model

Usually, the linear isotherm can only be assumed for very low solute concentrations.
At higher concentrations, the limited number of available binding sites on the surface of the adsorbent also needs to be considered.
The Langmuir equation takes this into account ([equations in CADET-Core](https://cadet.github.io/master/modelling/binding/multi_component_langmuir.html), [`Langmuir` in CADET-Process](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Langmuir.html)).

$$q = q_{sat} \cdot \frac{b \cdot c}{1 + b \cdot c} = \frac{a \cdot c}{1 + b \cdot c}$$

With:

- $q_{sat}$ = saturation loading, set by `capacity`
- $b$ = equilibrium factor, $b = k_a/k_d$ from `adsorption_rate` and `desorption_rate`
- $a = q_{sat} \cdot b$

Assumptions:

- All of the adsorption sites are equivalent, and each site can only accommodate one molecule.
- The surface is energetically homogeneous.
- Adsorbed molecules do not interact.
- There are no phase transitions.
- At the maximum adsorption, only a monolayer is formed.

A concentration step feeds two components into a `Cstr`. Both start at **0 mM** and have different binding strengths.

Both components are defined in one `ComponentSystem`.

In [ ]:
from CADETProcess.processModel import ComponentSystem

component_system = ComponentSystem(2)

We create the binding model with `Langmuir(component_system, name="langmuir")` and set one entry per component:

| Attribute | Value |
|---|---|
| `is_kinetic` | `False` |
| `adsorption_rate` | `[3, 1]` |
| `desorption_rate` | `[1, 1]` |
| `capacity` | `[1, 1]` mol/m³ |

In [ ]:
# Langmuir binding model


The binding model is attached to the tank, and an inlet feeds both components.

In [ ]:
from CADETProcess.processModel import Inlet, Outlet, Cstr

inlet = Inlet(component_system, name='inlet')
inlet.c = [1,1]
inlet.flow_rate = 1e-3

reactor = Cstr(component_system, name='reactor')
reactor.binding_model = binding_model

reactor.init_liquid_volume = 0.5*1e-3
reactor.const_solid_volume = 0.5*1e-3
reactor.c = [0.0, 0.0]
reactor.q = [0.0, 0.0]  # optional

reactor.solution_recorder.write_solution_bulk = True
reactor.solution_recorder.write_solution_solid = True

A `FlowSheet` connects the inlet to the tank.

In [ ]:
from CADETProcess.processModel import FlowSheet

flow_sheet = FlowSheet(component_system)

flow_sheet.add_unit(reactor)
flow_sheet.add_unit(inlet)

flow_sheet.add_connection(inlet,reactor)

A `Process` defines how long the concentration step runs.

In [ ]:
from CADETProcess.processModel import Process

process = Process(flow_sheet, 'process')
process.cycle_time = 10

`Cadet.simulate()` calculates the liquid and bound concentration histories.

In [ ]:
from CADETProcess.simulator import Cadet

simulator = Cadet()
simulator.time_resolution = 0.01

sim_results = simulator.simulate(process)

_ = sim_results.solution.reactor.bulk.plot()

The two components compete for binding sites. Their **bound concentrations over time** show how the different binding strengths affect uptake.

In [ ]:
# Show competitive uptake in the solid phase, alongside the liquid response above.
_ = sim_results.solution.reactor.solid.plot()
plt.show()

## Summary

- Linear binding relates liquid and bound concentrations through an equilibrium factor.
- Batch uptake data identify the equilibrium relation; uptake over time also contains kinetic information.
- In a column, binding delays a component according to $K$ and the solid fraction.
- Langmuir binding adds finite capacity and competition.

**Now:** **02.5 Binding: Exercises**: compare binding strength, capacity and kinetics in a column.

**Next lesson:** **03.1 Simple Chromatography** combines binding with a salt gradient.